# exp007 Feature Engineering Loop

## 0. Experiment Metadata

In [1]:
# ========================================
# EXPERIMENT CONFIG
# ========================================

EXP_NAME = "exp007_feature_engineering_loop"
TARGET = "Churn"
ID_COL = "id"

N_SPLITS = 5
SEED = 42

print(f"Running {EXP_NAME}")

Running exp007_feature_engineering_loop


## 1. Imports

In [2]:
# ========================================
# IMPORTS
# ========================================

import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import lightgbm as lgb
import xgboost as xgb

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',50)

## 2. Reproducibility

In [3]:
# ========================================
# SEED EVERYTHING
# ========================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

## 3. Load Data

In [4]:
# ========================================
# LOAD DATA
# ========================================

DATA_PATH = "/Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/data/raw/"

train = pd.read_csv(DATA_PATH + "train.csv")
test = pd.read_csv(DATA_PATH + "test.csv")

print(train.shape, test.shape)
train.head()

(594194, 21) (254655, 20)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


## 4. Quick Sanity Checs (Fast EDA)

In [5]:
# ========================================
# QUICK EDA
# ========================================

print("Target distribution:")
print(train['Churn'].value_counts(normalize=True))

print("\nMissing values (top 10):")
print(train.isnull().mean().sort_values(ascending=False).head(10))

Target distribution:
Churn
No     0.774792
Yes    0.225208
Name: proportion, dtype: float64

Missing values (top 10):
id                  0.0
DeviceProtection    0.0
TotalCharges        0.0
MonthlyCharges      0.0
PaymentMethod       0.0
PaperlessBilling    0.0
Contract            0.0
StreamingMovies     0.0
StreamingTV         0.0
TechSupport         0.0
dtype: float64


## 5. Feature Engineering 

In [17]:
def encode_data(X_train, X_test):
    X_train = X_train.copy()
    X_test = X_test.copy()

    categorical_cols = X_train.select_dtypes(include="object").columns.tolist()

    for col in categorical_cols:
        combined = pd.concat([X_train[col], X_test[col]], axis=0)

        combined_codes = combined.astype("category").cat.codes

        X_train[col] = combined_codes[:len(X_train)]
        X_test[col] = combined_codes[len(X_train):]

    return X_train, X_test

In [18]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import numpy as np

def evaluate_feature(train_df, test_df, feature_fn):

    # Apply feature engineering
    train_tmp = feature_fn(train_df.copy())
    test_tmp = feature_fn(test_df.copy())

    # ========================================
    # TARGET (ONLY TRAIN)
    # ========================================
    y = train_tmp[TARGET].map({"No": 0, "Yes": 1})

    # Remove ID + TARGET
    X = train_tmp.drop(columns=[TARGET, ID_COL])
    X_test = test_tmp.drop(columns=[ID_COL])

    # ========================================
    # ENCODE
    # ========================================
    X, X_test = encode_data(X, X_test)

    # ========================================
    # ALIGN
    # ========================================
    X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)

    # ========================================
    # FAST CV
    # ========================================
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

    oof = np.zeros(len(X))

    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            random_state=SEED
        )

        model.fit(X_tr, y_tr)
        oof[val_idx] = model.predict_proba(X_val)[:, 1]

    return roc_auc_score(y, oof)

In [24]:
# ========================================
# COMMON SETUP
# ========================================

service_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]


# ========================================
# FINANCIAL FEATURES (HIGH SIGNAL)
# ========================================

def feat_usage_velocity(df):
    if "MonthlyCharges" in df.columns and "tenure" in df.columns:
        df["usage_velocity"] = df["MonthlyCharges"] / (df["tenure"] + 1)
    return df


def feat_charge_diff(df):
    if all(col in df.columns for col in ["TotalCharges", "MonthlyCharges", "tenure"]):
        df["charge_diff"] = df["TotalCharges"] - (df["MonthlyCharges"] * df["tenure"])
    return df


def feat_avg_charge(df):
    if all(col in df.columns for col in ["TotalCharges", "tenure"]):
        df["avg_charge"] = df["TotalCharges"] / (df["tenure"] + 1)
    return df


# ========================================
# TENURE FEATURES (MEDIUM SIGNAL)
# ========================================

def feat_tenure_log(df):
    if "tenure" in df.columns:
        df["tenure_log"] = np.log1p(df["tenure"])
    return df


def feat_is_new(df):
    if "tenure" in df.columns:
        df["is_new"] = (df["tenure"] < 6).astype(int)
    return df


def feat_is_loyal(df):
    if "tenure" in df.columns:
        df["is_loyal"] = (df["tenure"] > 24).astype(int)
    return df


# ========================================
# SERVICE FEATURES (VERY IMPORTANT)
# ========================================

def feat_num_services(df):
    if all(col in df.columns for col in service_cols):
        df["num_services"] = (df[service_cols] == "Yes").sum(axis=1)
    return df


def feat_has_support(df):
    if "OnlineSecurity" in df.columns and "TechSupport" in df.columns:
        df["has_support"] = (
            (df["OnlineSecurity"] == "Yes") |
            (df["TechSupport"] == "Yes")
        ).astype(int)
    return df


# ========================================
# INTERACTION FEATURES (HIGH IMPACT)
# ========================================

def feat_high_charge_new(df):
    if "MonthlyCharges" in df.columns and "tenure" in df.columns:
        median = df["MonthlyCharges"].median()
        df["high_charge_new"] = (
            (df["MonthlyCharges"] > median) &
            (df["tenure"] < 12)
        ).astype(int)
    return df


def feat_service_per_tenure(df):
    if "tenure" in df.columns and all(col in df.columns for col in service_cols):
        num_services = (df[service_cols] == "Yes").sum(axis=1)
        df["service_per_tenure"] = num_services / (df["tenure"] + 1)
    return df


def feat_charge_per_service(df):
    if "MonthlyCharges" in df.columns and all(col in df.columns for col in service_cols):
        num_services = (df[service_cols] == "Yes").sum(axis=1)
        df["charge_per_service"] = df["MonthlyCharges"] / (num_services + 1)
    return df


# ========================================
# BUSINESS FEATURES (VERY HIGH SIGNAL)
# ========================================

def feat_contract_risk(df):
    if "Contract" in df.columns:
        mapping = {
            "Month-to-month": 2,
            "One year": 1,
            "Two year": 0
        }
        df["contract_risk"] = df["Contract"].map(mapping)
    return df


def feat_payment_risk(df):
    if "PaymentMethod" in df.columns:
        df["payment_risk"] = (df["PaymentMethod"] == "Electronic check").astype(int)
    return df


def feat_auto_pay(df):
    if "PaymentMethod" in df.columns:
        df["is_auto_pay"] = df["PaymentMethod"].isin([
            "Bank transfer (automatic)",
            "Credit card (automatic)"
        ]).astype(int)
    return df


# ========================================
# STRONG KAGGLE FEATURES (IMPORTANT ADD)
# ========================================

def feat_internet_type(df):
    if "InternetService" in df.columns:
        df["is_fiber"] = (df["InternetService"] == "Fiber optic").astype(int)
    return df


def feat_no_internet(df):
    if "InternetService" in df.columns:
        df["no_internet"] = (df["InternetService"] == "No").astype(int)
    return df


def feat_engagement(df):
    if "tenure" in df.columns and all(col in df.columns for col in service_cols):
        num_services = (df[service_cols] == "Yes").sum(axis=1)
        df["engagement"] = num_services * df["tenure"]
    return df


feature_functions = {
    # Strong core
    "usage_velocity": feat_usage_velocity,
    "num_services": feat_num_services,
    "charge_diff": feat_charge_diff,

    # High signal
    "contract_risk": feat_contract_risk,
    "payment_risk": feat_payment_risk,
    "auto_pay": feat_auto_pay,

    # Interaction (important)
    "high_charge_new": feat_high_charge_new,
    "service_per_tenure": feat_service_per_tenure,
    "charge_per_service": feat_charge_per_service,

    # Additional good ones
    "avg_charge": feat_avg_charge,
    "tenure_log": feat_tenure_log,
    "has_support": feat_has_support,

    # Kaggle-specific strong ones
    "is_fiber": feat_internet_type,
    "no_internet": feat_no_internet,
    "engagement": feat_engagement,
}

In [25]:
# ========================================
# BASELINE
# ========================================
baseline_score = evaluate_feature(train, test, lambda x: x)
print(f"Baseline AUC: {baseline_score:.5f}")

results = []

# ========================================
# LOOP
# ========================================
for name, fn in feature_functions.items():
    print(f"\nTesting: {name}")

    score = evaluate_feature(train, test, fn)
    gain = score - baseline_score

    print(f"AUC: {score:.5f} | Gain: {gain:.5f}")

    results.append({
        "feature": name,
        "score": score,
        "gain": gain
    })

results_df = pd.DataFrame(results).sort_values(by="gain", ascending=False)

print("\n=== FEATURE RANKING ===")
print(results_df)

[LightGBM] [Info] Number of positive: 89211, number of negative: 306918
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003886 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 626
[LightGBM] [Info] Number of data points in the train set: 396129, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225207 -> initscore=-1.235576
[LightGBM] [Info] Start training from score -1.235576
[LightGBM] [Info] Number of positive: 89211, number of negative: 306918
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003742 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 626
[LightGBM] [Info] Number of data points in the train set: 396129, number of used features: 19
[LightGBM] [Info

## 6. Target Encoding

In [7]:
# ========================================
# TARGET ENCODING (SIMPLE VERSION)
# ========================================

from category_encoders import TargetEncoder

# Get categorical columns from TRAIN
categorical_cols = train.select_dtypes(include="object").columns.tolist()

# Remove target column
categorical_cols = [col for col in categorical_cols if col != TARGET]

# Initialize encoder
encoder = TargetEncoder(cols=categorical_cols)

# Fit on train, transform both
train[categorical_cols] = encoder.fit_transform(train[categorical_cols], train[TARGET])
test[categorical_cols] = encoder.transform(test[categorical_cols])

## 7. Prepare Data

In [8]:
# ========================================
# PREPARE DATA
# ========================================

features = [col for col in train.columns if col not in [TARGET, ID_COL]]

train[TARGET] = train[TARGET].map({"No": 0, "Yes": 1}).astype(int)

X = train[features]
y = train[TARGET]

X_test = test[features]

## 8. Validation Strategy

In [9]:
# ========================================
# CV SETUP
# ========================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

## 9. Model Training

In [10]:
# ========================================
# TRAINING LOOP (LGB + XGB ENSEMBLE)
# ========================================

oof_preds_lgb = np.zeros(len(train))
oof_preds_xgb = np.zeros(len(train))

test_preds_lgb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))

scores_lgb = []
scores_xgb = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    
    print(f"\n--- Fold {fold} ---")
    
    X_train, X_valid = X.iloc[tr_idx], X.iloc[val_idx]
    y_train, y_valid = y.iloc[tr_idx], y.iloc[val_idx]

    # ======================
    # LIGHTGBM
    # ======================
    lgb_model = lgb.LGBMClassifier(
        n_estimators=5000,
        learning_rate=0.01,
        num_leaves=64,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=SEED,
        n_jobs=-1
    )

    lgb_model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(200)
        ]
    )

    val_pred_lgb = lgb_model.predict_proba(X_valid)[:, 1]
    oof_preds_lgb[val_idx] = val_pred_lgb
    test_preds_lgb += lgb_model.predict_proba(X_test)[:, 1] / N_SPLITS

    score_lgb = roc_auc_score(y_valid, val_pred_lgb)
    scores_lgb.append(score_lgb)

    # ======================
    # XGBOOST
    # ======================
    xgb_model = xgb.XGBClassifier(
        n_estimators=3000,
        learning_rate=0.01,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        eval_metric="auc",
        random_state=SEED,
        n_jobs=-1,
        tree_method="hist"
    )

    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False
    )

    val_pred_xgb = xgb_model.predict_proba(X_valid)[:, 1]
    oof_preds_xgb[val_idx] = val_pred_xgb
    test_preds_xgb += xgb_model.predict_proba(X_test)[:, 1] / N_SPLITS

    score_xgb = roc_auc_score(y_valid, val_pred_xgb)
    scores_xgb.append(score_xgb)

    print(f"LGB AUC: {score_lgb:.5f} | XGB AUC: {score_xgb:.5f}")


# ========================================
# CV RESULTS
# ========================================
print("\nLGB CV Mean:", np.mean(scores_lgb))
print("XGB CV Mean:", np.mean(scores_xgb))


# ========================================
# ENSEMBLE (FINAL)
# ========================================
oof_preds = 0.5 * oof_preds_lgb + 0.5 * oof_preds_xgb
test_preds = 0.5 * test_preds_lgb + 0.5 * test_preds_xgb

final_score = roc_auc_score(y, oof_preds)
print("\nEnsemble CV AUC:", final_score)


--- Fold 0 ---
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004004 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1158
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567
Training until validation scores don't improve for 100 rounds
[200]	valid_0's binary_logloss: 0.316516
[400]	valid_0's binary_logloss: 0.302372
[600]	valid_0's binary_logloss: 0.300278
[800]	valid_0's binary_logloss: 0.299565
[1000]	valid_0's binary_logloss: 0.299213
[1200]	valid_0's binary_logloss: 0.299023
[1400]	valid_0's binary_logloss: 0.298918
[1600]	valid_0's binary_logloss: 0.298832
[1800]	valid_0's binary_loglo

## 11. Feature Importance

In [113]:
# ========================================
# FEATURE IMPORTANCE
# ========================================

import matplotlib.pyplot as plt

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

importance.head(20)

plt.figure(figsize=(8, 10))
plt.barh(importance["feature"].head(20), importance["importance"].head(20))
plt.gca().invert_yaxis()
plt.title("Top Features")
plt.show()

ValueError: All arrays must be of the same length

## 12. Create Submission

In [12]:
# ========================================
# SUBMISSION (PROBABILITY - CORRECT)
# ========================================

import os

# Define path
OUTPUT_DIR = "/Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/outputs/submissions/"

# Create folder if not exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Create submission (USE PROBABILITY)
submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: test_preds   # ✅ probability, NOT binary
})

# File path
file_path = os.path.join(OUTPUT_DIR, f"{EXP_NAME}.csv")

# Save
submission.to_csv(file_path, index=False)

print(f"Submission saved at: {file_path}")

Submission saved at: /Users/theojeremiah/Workspace/01_DataScience/Projects/202603_Kaggle_CustomerChurn/outputs/submissions/exp006_feature_and_model_v2.csv
